# Notebook 07 - Embeddings, Semantic Search and RAG

Goes with **Lesson 08**. This is the heart of the Day 3 project.

In MOCK MODE, embeddings are replaced by word-count vectors (they match **words**, not deeper meaning) and answers are built from the best matching text. Everything still runs.

In [ ]:
import ai_helper
from ai_helper import ask_llm

ai_helper.status()

In [ ]:
import numpy as np
import pandas as pd
from ai_helper import embed, cosine_similarity

## 1. Embeddings: sentences become vectors

In [ ]:
sentences = [
    "How much is the fine for late books?",
    "The late fine for library books is 5 rupees per day.",
    "Jupiter is the largest planet and has a giant storm.",
    "Sleep 7 to 9 hours to remember what you study.",
    "Saturn has bright rings made of ice and rock.",
]
vectors = embed(sentences)
print("Number of vectors:", vectors.shape[0], " numbers in each vector:", vectors.shape[1])
print("First 8 numbers of the first vector:", vectors[0][:8].round(3))

## 2. Similarity between every pair of sentences

In [ ]:
labels = [s[:30] + "..." for s in sentences]
similarity = pd.DataFrame(vectors @ vectors.T, index=labels, columns=labels)
similarity.round(2)

In [ ]:
pairs = [(similarity.iloc[i, j], i, j) for i in range(len(sentences)) for j in range(i + 1, len(sentences))]
score, i, j = max(pairs)
print(f"Most similar pair ({score:.2f}):")
print("  ", sentences[i])
print("  ", sentences[j])

## 3. Semantic search over sentences

In [ ]:
def search(query, texts, text_vectors, top_k=2):
    query_vector = embed([query])[0]
    scores = text_vectors @ query_vector
    best = np.argsort(scores)[::-1][:top_k]
    return [(round(float(scores[k]), 3), texts[k]) for k in best]

for query in ["fine for returning books late", "which planet has rings?"]:
    print("QUERY:", query)
    for score, text in search(query, sentences, vectors):
        print(f"   {score}  {text}")

## 4. Chunking the documents

We split each document into chunks of about `chunk_size` characters, keeping whole lines together, with one line of overlap.

In [ ]:
from pathlib import Path

DOCS_DIR = Path("../data/documents") if Path("../data/documents").exists() else Path(".")

def chunk_text(text, chunk_size=400):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    # a line without a full stop is a heading: add ":" so it reads well when joined
    lines = [line if line[-1] in ".!?:" else line + ":" for line in lines]
    chunks, current = [], []
    for line in lines:
        if current and len(" ".join(current + [line])) > chunk_size:
            chunks.append(" ".join(current))
            current = current[-1:]          # overlap: repeat the last line in the next chunk
        current.append(line)
    if current:
        chunks.append(" ".join(current))
    return chunks

chunks = []
for file in sorted(DOCS_DIR.glob("*.txt")):
    for number, piece in enumerate(chunk_text(file.read_text(encoding="utf-8"))):
        chunks.append({"id": f"{file.stem}-{number}", "source": file.name, "text": piece})

print("Total chunks:", len(chunks))
for c in chunks[:3]:
    print()
    print(c["id"], "|", c["text"][:150], "...")

## 5. Build a tiny vector store (and save it)

A vector database stores **vector + text + metadata**. Ours is a list plus a numpy array. We also save it to a JSON file, which is what a vector database does on disk.

In [ ]:
import json

chunk_vectors = embed([c["text"] for c in chunks])
for c, v in zip(chunks, chunk_vectors):
    c["vector"] = v.tolist()

store_file = Path("my_vector_store.json")
store_file.write_text(json.dumps(chunks))
print(f"Saved {len(chunks)} chunks to {store_file} ({store_file.stat().st_size // 1024} KB)")

# Load it back, like a database would
store = json.loads(store_file.read_text())
store_vectors = np.array([c["vector"] for c in store])
print("Loaded vector store:", store_vectors.shape)

## 6. Retrieval and reranking

1. **Retrieve** the top 6 by vector similarity (fast, rough).
2. **Rerank**: add a bonus for chunks that share important words with the question, then keep the best 3.

In [ ]:
import re

STOP = {"the", "a", "an", "is", "of", "to", "for", "and", "what", "how", "in", "on", "if", "i", "my", "do", "does", "can", "it", "much", "many"}

def keywords(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP}

def retrieve(question, top_k=3, candidates=6):
    q_vec = embed([question])[0]
    scores = store_vectors @ q_vec
    first_round = np.argsort(scores)[::-1][:candidates]

    q_words = keywords(question)
    reranked = []
    for k in first_round:
        overlap = len(q_words & keywords(store[k]["text"])) / (len(q_words) or 1)
        final_score = 0.7 * float(scores[k]) + 0.3 * overlap
        reranked.append({**store[k], "score": round(final_score, 3), "vector_score": round(float(scores[k]), 3)})
    reranked.sort(key=lambda c: c["score"], reverse=True)
    return reranked[:top_k]

for hit in retrieve("What is the fine for a late book?"):
    print(f"{hit['score']:.3f} (vector {hit['vector_score']:.3f})  {hit['source']}: {hit['text'][:90]}...")

## 7. RAG: retrieve, then generate an answer with sources

In [ ]:
SYSTEM = ("You are a helpful study assistant. Answer using ONLY the context given. "
          "If the answer is not in the context, reply exactly: I could not find this in the documents. "
          "After each fact, write the source file name in brackets.")

def mock_rag_answer(question, hits):
    # Offline stand-in: quote the best matching sentences from the top chunk
    q_words = keywords(question)
    if not hits or hits[0]["score"] < 0.15 or not (q_words & keywords(hits[0]["text"])):
        return "I could not find this in the documents."
    sentences = re.split(r"(?<=[.!?])\s+", hits[0]["text"])
    best = sorted(sentences, key=lambda s: len(q_words & keywords(s)), reverse=True)[:2]
    return " ".join(best) + f" [{hits[0]['source']}]"

def rag_answer(question, top_k=3, show_prompt=False):
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(f"[{n + 1}] ({h['source']}) {h['text']}" for n, h in enumerate(hits))
    prompt = f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    if show_prompt:
        print("----- PROMPT SENT TO THE LLM -----")
        print(prompt)
        print("----------------------------------")
    answer = ask_llm(prompt, system=SYSTEM, temperature=0.1, mock_answer=mock_rag_answer(question, hits))
    sources = sorted({h["source"] for h in hits})
    return answer, sources

answer, sources = rag_answer("What is the fine for returning a book 3 days late?", show_prompt=True)
print()
print("ANSWER:", answer)
print("SOURCES SEARCHED:", sources)

In [ ]:
for question in ["How many books can a student borrow?",
                 "What is the Pomodoro method?",
                 "When did Chandrayaan-3 land?",
                 "Who won the cricket world cup in 2011?"]:
    answer, _ = rag_answer(question)
    print("Q:", question)
    print("A:", answer)
    print()

The last question is **not** in our documents, so a good RAG system says it could not find it instead of guessing.

## 8. RAG vs no RAG

Without the context, the model does not know our (made-up) college's rules.

In [ ]:
question = "What is the fine for returning a book late at Greenfield College library?"

print("WITHOUT RAG:")
print(ask_llm(question, temperature=0.1,
              mock_answer="I do not have specific information about Greenfield College's library fines. "
                          "Many libraries charge a small amount per day."))
print()
print("WITH RAG:")
print(rag_answer(question)[0])

## Try it yourself

1. Add your own `.txt` file to `data/documents/`, run the notebook again from the top (**Restart** then **Run All**), and ask about it.
2. Change `top_k` and `chunk_size`.

Go back to **Lesson 08** and do the **Check** section.